In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
SRMODELS   = CONFIGS['experiments']['sr']['optimizedeqs']
FIELDVARS  = CONFIGS['experiments']['nn']['runs']['nn_gauss']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SPLIT      = 'test'
NBINS      = 35
MINSAMPLES = 30
REGISTRY   = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),train_loss=row['train_loss'],valid_loss=row['valid_loss'])
              for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

In [ ]:
def kernel_integrate(fields,weights,dsig):
    return (fields*weights[None,:,:]*dsig[None,None,:]).sum(axis=2)

def calc_moisture_term(rh):
    return kappa*(rh-rh0)

def calc_instability_term(thetae,thetaestar):
    return thetae-gamma*thetaestar-theta0

def predict_from_sr_atm(moisture,instability):
    return np.maximum(np.expm1(alpha*np.maximum(moisture,instability)**3),0.0)

def bin_2d(x,y,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x,y,z  = x[finite],y[finite],z[finite]
    xedges = np.linspace(*np.percentile(x,[plo,phi]),nbins+1)
    yedges = np.linspace(*np.percentile(y,[plo,phi]),nbins+1)
    xi     = np.clip(np.digitize(x,xedges)-1,0,nbins-1)
    yi     = np.clip(np.digitize(y,yedges)-1,0,nbins-1)
    idx    = xi*nbins+yi
    counts = np.bincount(idx,minlength=nbins*nbins).reshape(nbins,nbins)
    sums   = np.bincount(idx,weights=z,minlength=nbins*nbins).reshape(nbins,nbins)
    return 0.5*(xedges[:-1]+xedges[1:]),0.5*(yedges[:-1]+yedges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpstd        = STATS['tp_std']
rhmean       = STATS['rh_mean']
rhstd        = STATS['rh_std']
thetaemean   = STATS['thetae_mean']
thetaestd    = STATS['thetae_std']
thetaesmean  = STATS['thetaestar_mean']
thetaesstd   = STATS['thetaestar_std']
c3,c4,c5     = (REGISTRY['sr_atm_eq']['constants'][name] for name in ['c3','c4','c5'])

alpha  = tpstd*c3/thetaestd**3
gamma  = c4*thetaestd/thetaesstd
theta0 = thetaemean-gamma*thetaesmean+c5*thetaestd
kappa  = thetaestd/rhstd
rh0    = rhmean

constdf = pd.DataFrame([
    {'Constant':'$\\alpha$','Value':f'{alpha:.3e}','Units':'K⁻³'},
    {'Constant':'$\\gamma$','Value':f'{gamma:.3f}','Units':'dimensionless'},
    {'Constant':'$\\Theta_0$','Value':f'{theta0:.2f}','Units':'K'},
    {'Constant':'$\\kappa$','Value':f'{kappa:.3f}','Units':'K/%'},
    {'Constant':'$\\mathrm{RH}_0$','Value':f'{rh0:.2f}','Units':'%'}])
display(constdf.style.hide(axis='index'))

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.sizes['time'],ds.sizes['lat'],ds.sizes['lon']
    nsig   = ds.sizes['sig']
    lat    = ds['lat'].values
    lon    = ds['lon'].values
    dsig   = ds['dsig'].values
    fields = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for name in FIELDVARS],axis=1)
    tp     = ds['tp'].transpose('time','lat','lon').values.ravel()
    lf     = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds['k'].values)
integrals = kernel_integrate(fields,np.mean(kernels,axis=0),dsig)
rh,thetae,thetaestar = (integrals[:,FIELDVARS.index(name)] for name in ['rh','thetae','thetaestar'])

moisture    = calc_moisture_term(rh)
instability = calc_instability_term(thetae,thetaestar)
predtp      = predict_from_sr_atm(moisture,instability)
valid       = np.isfinite(moisture)&np.isfinite(instability)&np.isfinite(tp)

with xr.open_dataset(os.path.join(PREDSDIR,f'sr_atm_eq_{SPLIT}_predictions.nc')) as ds:
    savedtp = ds['tp'].squeeze().transpose('time','lat','lon').values.ravel()
maxdiff = np.nanmax(np.abs(predtp[valid]-savedtp[valid]))
print(f'Max |physical-form prediction - saved SR-ATM prediction| = {maxdiff:.2e} mm')

In [ ]:
moistdom = moisture>=instability
rows = []
for region,regionmask in [('All',valid),('Land',valid&(lf>=0.5)),('Ocean',valid&(lf<0.5))]:
    for regime,regimemask in [('Moisture-dominated',moistdom),('Instability-dominated',~moistdom)]:
        selected = regionmask&regimemask
        rows.append({'Region':region,'Regime':regime,
                     'Fraction of Samples':100*selected.sum()/regionmask.sum(),
                     'Mean ERA5 P (mm)':tp[selected].mean(),
                     'Mean SR-ATM P (mm)':predtp[selected].mean()})
regimedf = pd.DataFrame(rows)
display(regimedf.style.hide(axis='index').format({'Fraction of Samples':'{:.1f}%','Mean ERA5 P (mm)':'{:.3f}','Mean SR-ATM P (mm)':'{:.3f}'}))

In [ ]:
M,I,P   = moisture[valid],instability[valid],tp[valid]
Mlo,Mhi = np.percentile(M,[1,99])
Ilo,Ihi = np.percentile(I,[1,99])
axlo    = min(Mlo,Ilo)
axhi    = max(Mhi,Ihi)
xc,yc,obsbin,_ = bin_2d(M,I,P)
density = np.log10(np.histogram2d(M,I,bins=[np.linspace(Mlo,Mhi,NBINS+1),np.linspace(Ilo,Ihi,NBINS+1)])[0].T.clip(1))
Mgrid,Igrid = np.meshgrid(np.linspace(Mlo,Mhi,200),np.linspace(Ilo,Ihi,200))
predgrid = predict_from_sr_atm(Mgrid,Igrid)
fracmap  = np.nanmean(np.where(valid,moistdom,np.nan).reshape(ntime,nlat,nlon),axis=0)

xlabel = 'Moisture Term (K)'
ylabel = 'Instability Term (K)'
kwprecip = dict(cmap='ColdHot_r',cmap_kw={'left':0.5},vmin=0,vmax=4,levels=18,extend='max')
kwmap    = dict(coast=True,lonlim=LONRANGE,lonlines=[65,75,85],lonlabels='b',
                latlim=LATRANGE,latlines=[10,15,20],latlabels='l',grid=False)

fig,axs = pplt.subplots([[1,2],[3,4]],figwidth=5.5,proj={4:'cyl'},share=False,hspace=8)
m0 = axs[0].pcolormesh(np.linspace(Mlo,Mhi,200),np.linspace(Ilo,Ihi,200),predgrid,**kwprecip)
axs[0].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
axs[0].text(Mhi-0.13*(Mhi-Mlo),Ilo+0.08*(Ihi-Ilo),'moisture-dominated',ha='right',va='bottom')
axs[0].text(Mlo+0.05*(Mhi-Mlo),Ihi-0.12*(Ihi-Ilo),'instability-dominated',ha='left',va='top')
axs[0].format(xlabel=xlabel,ylabel=ylabel,title='SR-ATM',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
axs[1].pcolormesh(xc,yc,obsbin.T,**kwprecip)
axs[1].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
axs[1].format(xlabel=xlabel,ylabel='',yticklabels=[],title='ERA5',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi),facecolor='gray1')
m2 = axs[2].pcolormesh(xc,yc,density,cmap='Grays',vmin=0)
axs[2].plot([axlo,axhi],[axlo,axhi],'k--',lw=0.8)
axs[2].format(xlabel=xlabel,ylabel=ylabel,title='Sample Density',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
axs[2].colorbar(m2,loc='b',label='log$_{10}$($\\mathit{N}$)')
m3 = axs[3].pcolormesh(lon,lat,fracmap,cmap='ColdHot_r',vmin=0,vmax=1)
axs[3].format(title='Moisture-Dominated Fraction',**kwmap)
axs[3].colorbar(m3,loc='b',ticks=0.2,label='Fraction of Timesteps')
fig.format(abc=True,titleloc='l')
fig.canvas.draw()
figh  = fig.get_figheight()
left  = min(axs[0].get_position().x0,axs[1].get_position().x0)
right = max(axs[0].get_position().x1,axs[1].get_position().x1)
btop  = min(axs[0].get_position().y0,axs[1].get_position().y0)
tbot  = max(axs[2].get_position().y1,axs[3].get_position().y1)
cax   = fig.add_axes([left,tbot+0.5*(btop-tbot)-0.09/figh,right-left,0.18/figh])
fig.colorbar(m0,cax=cax,orientation='horizontal',label='Total Precipitation (mm)',extend='max')
pplt.show()
fig.save('../figs/fig_4.jpg')